# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q esda libpysal splot
    print('Packages installed.')

# Spatial Autocorrelation (Moran's I)

Spatial autocorrelation measures how correlated a variable is with itself in space. 
-   **Positive**: High values cluster near high values (Hotspots).
-   **Negative**: High values are near low values (Checkerboard).
-   **Zero**: Random distribution.

We will use **Moran's I** to quantify this on a synthetic grid.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import libpysal 
from libpysal.weights import lat2W
from esda.moran import Moran
from splot.esda import plot_moran, moran_scatterplot

# Fix for reproducibility
np.random.seed(42)

## 1. Generate Synthetic Data

We create a 10x10 grid (100 units). We'll generate two patterns:
1.  **Random**: Pure noise.
2.  **Clustered**: Spatially dependent data.

In [ ]:
size = 10
n = size * size

# 1. Random Data
y_random = np.random.rand(n)

# 2. Clustered Data (Smooth Gradient + Noise)
x_coords = np.tile(np.arange(size), size)
y_coords = np.repeat(np.arange(size), size)
# Value increases with X and Y
y_clustered = x_coords + y_coords + np.random.normal(0, 2, n)

# Reshape for visualization
grid_random = y_random.reshape((size, size))
grid_clustered = y_clustered.reshape((size, size))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(grid_random, ax=ax[0], cbar=False, cmap='viridis')
ax[0].set_title("Random Data")

sns.heatmap(grid_clustered, ax=ax[1], cbar=False, cmap='viridis')
ax[1].set_title("Clustered Data")
plt.show()

## 2. Create Spatial Weights Matrix

We need to define "who is neighbor to whom". For a grid, we use Rook contiguity (sharing an edge).

In [ ]:
w = lat2W(size, size)
w.transform = 'r' # Row-standardize
print(f"Neighbors for cell 0: {w.neighbors[0]}")

## 3. Calculate Moran's I

Range: -1 (Dispersed) to +1 (Clustered). 0 is Random.

In [ ]:
mi_random = Moran(y_random, w)
mi_clustered = Moran(y_clustered, w)

print(f"Random Data Moran's I: {mi_random.I:.3f} (p-value: {mi_random.p_sim:.3f})")
print(f"Clustered Data Moran's I: {mi_clustered.I:.3f} (p-value: {mi_clustered.p_sim:.3f})")

## 4. Moran Scatterplot

Visualizing the relationship between a value and the average of its neighbors (Lag).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

moran_scatterplot(mi_random, ax=ax[0])
ax[0].set_title(f"Random (I={mi_random.I:.2f})")

moran_scatterplot(mi_clustered, ax=ax[1])
ax[1].set_title(f"Clustered (I={mi_clustered.I:.2f})")

plt.show()